In [1]:
import gc
import os
import sys

import numpy as np
import optuna
import pandas as pd
import torch
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# ruff: noqa
%load_ext autoreload
%autoreload 2
from drGAT import No_atten_drGAT
from drGAT.load_data import load_data
from drGAT.sampler import RandomSampler

/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/torch_geometric/typing.py:54: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /lib64/libm.so.6: version `GLIBC_2.29' not found (required by /gpfs/gsfs12/users/inouey2/conda/envs/genex/lib/python3.10/site-packages/libpyg.so)
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/torch_geometric/typing.py:72: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /gpfs/gsfs12/users/inouey2/conda/envs/genex/lib/python3.10/site-packages/torch_scatter/_scatter_cuda.so: undefined symbol: _ZN2at23SavedTensorDefaultHooks11set_tracingEb
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/torch_geometric/typing.py:99: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. S

In [3]:
method = "MPNN"
name = "NCI"
data = "nci"
study = optuna.create_study(
    storage="sqlite:///{}_{}.sqlite3".format(name, "GCN_MPNN"),
    study_name=name,
    load_if_exists=True,
)
df = study.trials_dataframe()
df = df.dropna(subset=[i for i in df.columns if "values" in i])
tmp = df.loc[
    df[["values_0", "values_1", "values_2", "values_3"]]
    .max(axis=1)
    .sort_values(ascending=False)
    .index
]
params = tmp[tmp["params_gnn_layer"] == method].head().iloc[0]
print(params)
params = {
    i.replace("params_", ""): j
    for i, j in zip(pd.DataFrame(params).index, params)
    if "params" in i
}

[I 2025-04-02 22:39:34,938] Using an existing study with name 'NCI' instead of creating a new one.


number                                            644
values_0                                     0.506682
values_1                                     0.514177
values_2                                     0.499915
values_3                                     0.668069
datetime_start             2025-03-28 09:11:08.123751
datetime_complete          2025-03-28 09:11:27.371461
duration                       0 days 00:00:19.247710
params_T_max                                      NaN
params_activation                                relu
params_amsgrad                                   True
params_dropout1                                   0.1
params_dropout2                                   0.2
params_gamma_step                                 NaN
params_gnn_layer                                 MPNN
params_heads                                        2
params_hidden1                                    512
params_hidden2                                    256
params_hidden3              

In [4]:
drugAct, pos_num, null_mask, S_d, S_c, S_g, A_cg, A_dg = load_data(data)

load nci
unique drugs: 177
unique genes: 251
DTI unique genes:  251
Top 90% variable genes:  2383
Total:  2582
Final gene exp shape: (60, 2582)
Final drug Act shape: (1005, 60)


100%|██████████| 25/25 [00:02<00:00, 11.19it/s]


Done!


In [5]:
import math


def auto_convert_params(params, nan_replace=None):
    """パラメータの型を自動変換する関数

    Args:
        params (dict): 変換前のパラメータ辞書
        nan_replace: NaNの置換値（デフォルトはNone）

    Returns:
        dict: 型変換後のパラメータ辞書
    """
    converted = {}
    for k, v in params.items():
        # NaNの処理
        if isinstance(v, float) and math.isnan(v):
            converted[k] = nan_replace
        # 浮動小数点数 → 整数変換（例: 1028.0 → 1028）
        elif isinstance(v, float) and v.is_integer():
            converted[k] = int(v)
        # その他の値はそのまま保持
        else:
            converted[k] = v
    return converted

In [6]:
params = auto_convert_params(params, nan_replace=0)

In [17]:
params.update(
    {
        "n_drug": S_d.shape[0],
        "n_cell": S_c.shape[0],
        "n_gene": S_g.shape[0],
        "epochs": 1000,
        "lr": 0.0005,
    }
)

In [18]:
params

{'T_max': 0,
 'activation': 'relu',
 'amsgrad': True,
 'dropout1': 0.1,
 'dropout2': 0.2,
 'gamma_step': 0,
 'gnn_layer': 'MPNN',
 'heads': 2,
 'hidden1': 512,
 'hidden2': 256,
 'hidden3': 64,
 'lr': 0.0005,
 'momentum': 0,
 'nesterov': 0,
 'optimizer': 'Adam',
 'patience_plateau': 9,
 'scheduler': 'Plateau',
 'step_size': 0,
 'thresh_plateau': 0.00013663049963442048,
 'weight_decay': 1.2914099787395681e-05,
 'n_drug': 1005,
 'n_cell': 60,
 'n_gene': 2582,
 'epochs': 1000}

In [19]:
PATH = f"../{data}_data/"

In [20]:
# K-fold cross validation
k = 5
kfold = KFold(n_splits=k, shuffle=True, random_state=42)

true_datas = pd.DataFrame()
predict_datas = pd.DataFrame()

for train_index, test_index in kfold.split(np.arange(pos_num)):
    sampler = RandomSampler(
        drugAct,
        train_index,
        test_index,
        null_mask,
        S_d,
        S_c,
        S_g,
        A_cg,
        A_dg,
        PATH,
    )
    (_, best_val_labels, best_val_prob, best_metrics, _, _, _) = No_atten_drGAT.train(
        sampler, params=params, device=device, verbose=False
    )
    true_datas = pd.concat(
        [true_datas, pd.DataFrame(best_val_labels)],
        ignore_index=True,
        axis=1,
    )
    predict_datas = pd.concat(
        [predict_datas, pd.DataFrame(best_val_prob)],
        ignore_index=True,
        axis=1,
    )

true_datas.to_csv(f"true_{data}_{method}.csv")
predict_datas.to_csv(f"pred_{data}_{method}.csv")

Using device: cuda


  6%|▌         | 58/1000 [00:28<07:46,  2.02it/s]

KeyboardInterrupt



In [ ]:
true_datas

In [ ]:
predict_datas